# Advanced Python Aggregators — Problems With Solutions

This notebook expands the topic of **aggregators**:

- `min`, `max`, `sum`
- `any`, `all`
- `map` and generator expressions
- iterator exhaustion
- `key=` and `default=`
- streaming-style validation
- custom reductions with `functools.reduce`

Each problem includes a solution, tests, and extra examples.

## Best-practice reminders

1. Prefer **generator expressions** when you only need one pass.
2. Remember that aggregators **consume iterators**.
3. Use `key=` with `min` / `max` instead of sorting when you only need one best item.
4. Use `default=` with `min` / `max` when an iterable may be empty.
5. Use `any` / `all` for predicate checks because they **short-circuit**.
6. Avoid materializing large iterables unless you truly need the whole list.

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from decimal import Decimal
from fractions import Fraction
from functools import reduce
from itertools import count, islice, tee
from math import isfinite
from numbers import Number
from operator import mul
from pathlib import Path
from typing import Callable, Iterable, Iterator, Any

print("Setup complete")

Setup complete


## Setup: sample iterables and tiny data files

The notebook is self-contained. The next cell creates data used by later problems.

In [2]:
def squares(n: int) -> Iterator[int]:
    """Yield square numbers lazily."""
    for i in range(n):
        yield i ** 2


car_brands = [
    "Alfa Romeo", "Aston Martin", "Audi", "Bentley", "Benz", "BMW",
    "Bugatti", "Cadillac", "Chevrolet", "Chrysler", "Citroen",
    "Corvette", "DAF", "Dacia", "Daewoo", "Daihatsu", "Datsun",
    "De Lorean", "Dino", "Dodge"
]

Path("car-brands.txt").write_text("\n".join(car_brands), encoding="utf-8")

transactions_csv = """id,account,amount,currency,status
1,A100,250.50,USD,posted
2,A100,-10.25,USD,posted
3,A200,999.99,EUR,pending
4,A200,-1200.00,EUR,posted
5,A300,0.00,USD,failed
6,A100,75.00,USD,posted
7,A400,5000.00,GBP,posted
8,A400,-4999.99,GBP,posted
"""
Path("transactions.csv").write_text(transactions_csv, encoding="utf-8")

log_lines = [
    "2026-07-03T09:00:01 INFO latency_ms=12 endpoint=/health",
    "2026-07-03T09:00:02 INFO latency_ms=44 endpoint=/api/users",
    "2026-07-03T09:00:03 WARNING latency_ms=230 endpoint=/api/orders",
    "2026-07-03T09:00:04 ERROR latency_ms=891 endpoint=/api/orders",
    "2026-07-03T09:00:05 INFO latency_ms=31 endpoint=/api/users",
]
Path("app.log").write_text("\n".join(log_lines), encoding="utf-8")

list(squares(5)), Path("car-brands.txt").exists(), Path("transactions.csv").exists()

([0, 1, 4, 9, 16], True, True)

## Example A: basic aggregators

In [3]:
values = list(squares(8))

print("values:", values)
print("min:", min(values))
print("max:", max(values))
print("sum:", sum(values))
print("count:", len(values))
print("mean:", sum(values) / len(values))

values: [0, 1, 4, 9, 16, 25, 36, 49]
min: 0
max: 49
sum: 140
count: 8
mean: 17.5


## Example B: iterator exhaustion

Aggregators consume an iterator. Once `max(sq)` runs, the iterator is empty.

In [4]:
sq = squares(5)

print("First aggregation:", max(sq))

try:
    print("Second aggregation:", min(sq))
except ValueError as ex:
    print(type(ex).__name__, "-", ex)

First aggregation: 16
ValueError - min() iterable argument is empty


## Example C: truthiness, `any`, and `all`

- `any(iterable)` is like asking: “Is at least one item truthy?”
- `all(iterable)` is like asking: “Are all items truthy?”
- Both short-circuit.

In [5]:
examples = [
    [0, "", None],
    [0, "", None, "hello"],
    [1, "abc", [1, 2], range(3)],
    [1, "abc", [], range(3)],
]

for items in examples:
    print(f"{items!r:35} any={any(items)!s:5} all={all(items)!s:5}")

[0, '', None]                       any=False all=False
[0, '', None, 'hello']              any=True  all=False
[1, 'abc', [1, 2], range(0, 3)]     any=True  all=True 
[1, 'abc', [], range(0, 3)]         any=True  all=False


## Problem 1 — Numeric validation, but handle `bool` correctly

### Task

Write `all_numeric(items, allow_bool=False)`.

Requirements:

- Return `True` only if every item is numeric.
- `int`, `float`, `complex`, `Decimal`, and `Fraction` should count as numeric.
- `bool` is a subclass of `int`, so decide whether to include it with `allow_bool`.
- Use `all`.

In [6]:
# Solution

def all_numeric(items: Iterable[Any], *, allow_bool: bool = False) -> bool:
    def is_allowed_number(x: Any) -> bool:
        if isinstance(x, bool) and not allow_bool:
            return False
        return isinstance(x, Number)

    return all(is_allowed_number(x) for x in items)


tests = [
    ([10, 20, Decimal("3.14"), Fraction(1, 3)], False),
    ([10, True, 20], False),
    ([10, True, 20], True),
    ([10, "20", 30], False),
    ([], False),  # all([]) is True: vacuous truth
]

for values, allow_bool in tests:
    print(f"{values!r:45} allow_bool={allow_bool:<5} -> {all_numeric(values, allow_bool=allow_bool)}")

[10, 20, Decimal('3.14'), Fraction(1, 3)]     allow_bool=0     -> True
[10, True, 20]                                allow_bool=0     -> False
[10, True, 20]                                allow_bool=1     -> True
[10, '20', 30]                                allow_bool=0     -> False
[]                                            allow_bool=0     -> True


### Extra examples for Problem 1

Notice that `all([])` returns `True`. This is mathematically useful, but it may be surprising in validation code.

In [7]:
print("all([]):", all([]))
print("any([]):", any([]))

def all_numeric_non_empty(items: Iterable[Any], *, allow_bool: bool = False) -> bool:
    materialized = list(items)
    return bool(materialized) and all_numeric(materialized, allow_bool=allow_bool)

print(all_numeric_non_empty([]))
print(all_numeric_non_empty([1, 2, 3]))

all([]): True
any([]): False
False
True


## Problem 2 — Short-circuiting with expensive predicates

### Task

Show that `any` and `all` stop as soon as the final answer is known.

Build a predicate that prints when it is called, then compare:

- `any(predicate(x) for x in data)`
- `any([predicate(x) for x in data])`

The first version should stop early. The second version should evaluate the whole list first.

In [8]:
# Solution

def is_large_with_trace(x: int) -> bool:
    print(f"checking {x}")
    return x > 5

data = [1, 2, 3, 9, 10, 11]

print("Generator expression:")
result = any(is_large_with_trace(x) for x in data)
print("result:", result)

print("\nList comprehension:")
result = any([is_large_with_trace(x) for x in data])
print("result:", result)

Generator expression:
checking 1
checking 2
checking 3
checking 9
result: True

List comprehension:
checking 1
checking 2
checking 3
checking 9
checking 10
checking 11
result: True


### Best practice

For `any` / `all`, prefer a generator expression unless you need the intermediate list.

In [9]:
def has_negative(values: Iterable[int]) -> bool:
    return any(x < 0 for x in values)

def all_between_0_and_100(values: Iterable[int]) -> bool:
    return all(0 <= x <= 100 for x in values)

print(has_negative([5, 3, -1, 10]))
print(all_between_0_and_100([0, 50, 100]))
print(all_between_0_and_100([0, 50, 101]))

True
True
False


## Problem 3 — Safe `min` / `max` with `default=`

### Task

Write `smallest_positive(values)`.

Requirements:

- Return the smallest value greater than `0`.
- Return `None` if no positive value exists.
- Do not build a temporary list.

In [10]:
# Solution

def smallest_positive(values: Iterable[float]) -> float | None:
    return min((x for x in values if x > 0), default=None)


print(smallest_positive([0, -5, 10, 2, 99]))
print(smallest_positive([-10, -1, 0]))
print(smallest_positive([]))

2
None
None


### Extra examples: `key=` for best item selection

Use `max(items, key=...)` instead of sorting all items when you only need one item.

In [11]:
@dataclass(frozen=True)
class Product:
    name: str
    price: Decimal
    rating: float
    reviews: int


products = [
    Product("Keyboard", Decimal("89.99"), 4.7, 1200),
    Product("Mouse", Decimal("45.50"), 4.5, 980),
    Product("Monitor", Decimal("299.99"), 4.8, 650),
    Product("USB Hub", Decimal("25.00"), 4.6, 2200),
]

best_rating = max(products, key=lambda p: p.rating)
most_reviewed = max(products, key=lambda p: p.reviews)
best_value = max(products, key=lambda p: (p.rating * p.reviews) / float(p.price))

print("best_rating:", best_rating)
print("most_reviewed:", most_reviewed)
print("best_value:", best_value)

best_rating: Product(name='Monitor', price=Decimal('299.99'), rating=4.8, reviews=650)
most_reviewed: Product(name='USB Hub', price=Decimal('25.00'), rating=4.6, reviews=2200)
best_value: Product(name='USB Hub', price=Decimal('25.00'), rating=4.6, reviews=2200)


## Problem 4 — Parse a CSV file and aggregate posted transactions

### Task

Using `transactions.csv`, compute:

1. Total posted amount per currency.
2. Largest absolute posted transaction.
3. Whether all posted transactions have non-empty account IDs.
4. Whether any failed transaction has amount `0`.

In [12]:
# Solution

def parse_transactions(path: str | Path) -> Iterator[dict[str, str]]:
    with open(path, encoding="utf-8") as f:
        header = next(f).strip().split(",")
        for line in f:
            values = line.strip().split(",")
            yield dict(zip(header, values))


def posted_transactions(path: str | Path) -> Iterator[dict[str, str]]:
    return (row for row in parse_transactions(path) if row["status"] == "posted")


totals = defaultdict(Decimal)

for row in posted_transactions("transactions.csv"):
    totals[row["currency"]] += Decimal(row["amount"])

largest_abs = max(
    posted_transactions("transactions.csv"),
    key=lambda row: abs(Decimal(row["amount"])),
    default=None
)

all_have_account = all(row["account"] for row in posted_transactions("transactions.csv"))

any_failed_zero = any(
    row["status"] == "failed" and Decimal(row["amount"]) == 0
    for row in parse_transactions("transactions.csv")
)

print("totals:", dict(totals))
print("largest_abs:", largest_abs)
print("all_have_account:", all_have_account)
print("any_failed_zero:", any_failed_zero)

totals: {'USD': Decimal('315.25'), 'EUR': Decimal('-1200.00'), 'GBP': Decimal('0.01')}
largest_abs: {'id': '7', 'account': 'A400', 'amount': '5000.00', 'currency': 'GBP', 'status': 'posted'}
all_have_account: True
any_failed_zero: True


### Extra example: one-pass aggregation

When the input is large, avoid reading it multiple times. Build a small summary object in one pass.

In [13]:
@dataclass
class TransactionSummary:
    totals: dict[str, Decimal]
    largest_abs: dict[str, str] | None
    posted_count: int
    failed_zero_count: int
    bad_account_count: int


def summarize_transactions(path: str | Path) -> TransactionSummary:
    totals = defaultdict(Decimal)
    largest_abs = None
    posted_count = 0
    failed_zero_count = 0
    bad_account_count = 0

    for row in parse_transactions(path):
        amount = Decimal(row["amount"])

        if row["status"] == "posted":
            posted_count += 1
            totals[row["currency"]] += amount

            if not row["account"]:
                bad_account_count += 1

            if largest_abs is None or abs(amount) > abs(Decimal(largest_abs["amount"])):
                largest_abs = row

        if row["status"] == "failed" and amount == 0:
            failed_zero_count += 1

    return TransactionSummary(
        totals=dict(totals),
        largest_abs=largest_abs,
        posted_count=posted_count,
        failed_zero_count=failed_zero_count,
        bad_account_count=bad_account_count,
    )


summary = summarize_transactions("transactions.csv")
summary

TransactionSummary(totals={'USD': Decimal('315.25'), 'EUR': Decimal('-1200.00'), 'GBP': Decimal('0.01')}, largest_abs={'id': '7', 'account': 'A400', 'amount': '5000.00', 'currency': 'GBP', 'status': 'posted'}, posted_count=6, failed_zero_count=1, bad_account_count=0)

## Problem 5 — Log-file aggregation

### Task

Using `app.log`, compute:

1. Number of `ERROR` lines.
2. Maximum latency.
3. Whether all latencies are finite and non-negative.
4. Whether any endpoint has latency above `500 ms`.
5. The endpoint with the highest latency.

In [14]:
# Solution

def parse_log_line(line: str) -> dict[str, Any]:
    parts = line.split()
    timestamp, level = parts[0], parts[1]

    fields = {}
    for part in parts[2:]:
        key, value = part.split("=", 1)
        fields[key] = value

    return {
        "timestamp": timestamp,
        "level": level,
        "latency_ms": float(fields["latency_ms"]),
        "endpoint": fields["endpoint"],
    }


def read_logs(path: str | Path) -> Iterator[dict[str, Any]]:
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield parse_log_line(line.strip())


error_count = sum(1 for row in read_logs("app.log") if row["level"] == "ERROR")
max_latency = max((row["latency_ms"] for row in read_logs("app.log")), default=None)
latencies_ok = all(isfinite(row["latency_ms"]) and row["latency_ms"] >= 0 for row in read_logs("app.log"))
any_slow = any(row["latency_ms"] > 500 for row in read_logs("app.log"))
slowest = max(read_logs("app.log"), key=lambda row: row["latency_ms"], default=None)

print("error_count:", error_count)
print("max_latency:", max_latency)
print("latencies_ok:", latencies_ok)
print("any_slow:", any_slow)
print("slowest:", slowest)

error_count: 1
max_latency: 891.0
latencies_ok: True
any_slow: True
slowest: {'timestamp': '2026-07-03T09:00:04', 'level': 'ERROR', 'latency_ms': 891.0, 'endpoint': '/api/orders'}


### Extra example: count levels with `Counter`

In [15]:
level_counts = Counter(row["level"] for row in read_logs("app.log"))
endpoint_counts = Counter(row["endpoint"] for row in read_logs("app.log"))

print(level_counts)
print(endpoint_counts)
print("most common endpoint:", endpoint_counts.most_common(1))

Counter({'INFO': 3, 'WARNING': 1, 'ERROR': 1})
Counter({'/api/users': 2, '/api/orders': 2, '/health': 1})
most common endpoint: [('/api/users', 2)]


## Problem 6 — Avoid a common bug: using the same iterator twice

### Task

The following code is buggy:

```python
nums = (x for x in [10, 20, 30])
print(sum(nums))
print(max(nums))
```

Fix it in three different ways.

In [16]:
# Solution 1: recreate the generator each time

def numbers() -> Iterator[int]:
    yield from [10, 20, 30]

print("sum:", sum(numbers()))
print("max:", max(numbers()))

sum: 60
max: 30


In [17]:
# Solution 2: materialize the data if it is small enough

nums = list(x for x in [10, 20, 30])

print("sum:", sum(nums))
print("max:", max(nums))

sum: 60
max: 30


In [18]:
# Solution 3: use itertools.tee carefully

nums1, nums2 = tee((x for x in [10, 20, 30]), 2)

print("sum:", sum(nums1))
print("max:", max(nums2))

sum: 60
max: 30


### Best-practice note on `tee`

`tee` can cache values internally. It is useful, but it is not always ideal for huge or infinite streams.

## Problem 7 — Custom aggregation with `reduce`

### Task

Implement product aggregation in two ways:

1. With a normal loop.
2. With `functools.reduce`.

Then compare with `sum`, which is also a specialized aggregation.

In [19]:
# Solution

def product_loop(values: Iterable[int | float]) -> int | float:
    result = 1
    for value in values:
        result *= value
    return result


def product_reduce(values: Iterable[int | float]) -> int | float:
    return reduce(mul, values, 1)


values = [2, 3, 4, 5]

print("loop product:", product_loop(values))
print("reduce product:", product_reduce(values))
print("sum:", sum(values))

loop product: 120
reduce product: 120
sum: 14


### Extra example: reduce for merging dictionaries

Often a loop is clearer than `reduce`, but seeing the pattern is useful.

In [20]:
records = [
    {"apples": 3, "bananas": 1},
    {"apples": 2, "oranges": 5},
    {"bananas": 4, "oranges": 1},
]

def merge_counts(left: dict[str, int], right: dict[str, int]) -> dict[str, int]:
    merged = dict(left)
    for key, value in right.items():
        merged[key] = merged.get(key, 0) + value
    return merged

print(reduce(merge_counts, records, {}))

# Usually clearer:
counts = Counter()
for record in records:
    counts.update(record)

print(dict(counts))

{'apples': 5, 'bananas': 5, 'oranges': 6}
{'apples': 5, 'bananas': 5, 'oranges': 6}


## Problem 8 — Find the first item matching a condition

### Task

There is no built-in `first` aggregator, but we can combine `next` with a generator expression.

Write `first_match(items, predicate, default=None)`.

In [21]:
# Solution

def first_match(
    items: Iterable[Any],
    predicate: Callable[[Any], bool],
    default: Any = None,
) -> Any:
    return next((item for item in items if predicate(item)), default)


numbers = [3, 5, 7, 8, 11]
print(first_match(numbers, lambda x: x % 2 == 0))
print(first_match(numbers, lambda x: x > 100, default="not found"))

first_long_brand = first_match(car_brands, lambda name: len(name) >= 10)
print(first_long_brand)

8
not found
Alfa Romeo


### Extra example: `next` can also consume iterators

Like `min`, `max`, `sum`, `any`, and `all`, `next` advances an iterator.

In [22]:
it = iter([10, 20, 30])

print(next(it))
print(next(it))
print(list(it))

10
20
[30]


## Problem 9 — Validate nested records with `all` and `any`

### Task

Given user records, validate:

1. All users have an integer `id`.
2. All users have at least one role.
3. Any user is an admin.
4. No user has an empty email.
5. Every email contains `@`.

In [23]:
# Solution

users = [
    {"id": 1, "email": "ana@example.com", "roles": ["reader"]},
    {"id": 2, "email": "bo@example.com", "roles": ["reader", "admin"]},
    {"id": 3, "email": "cy@example.com", "roles": ["editor"]},
]

all_integer_ids = all(isinstance(user.get("id"), int) for user in users)
all_have_roles = all(bool(user.get("roles")) for user in users)
any_admin = any("admin" in user.get("roles", []) for user in users)
no_empty_email = not any(not user.get("email") for user in users)
all_valid_email_shape = all("@" in user.get("email", "") for user in users)

print("all_integer_ids:", all_integer_ids)
print("all_have_roles:", all_have_roles)
print("any_admin:", any_admin)
print("no_empty_email:", no_empty_email)
print("all_valid_email_shape:", all_valid_email_shape)

all_integer_ids: True
all_have_roles: True
any_admin: True
no_empty_email: True
all_valid_email_shape: True


### Extra example: collect validation errors

Aggregators tell you if validation passes. Sometimes you also need error messages.

In [24]:
def validate_user(user: dict[str, Any]) -> list[str]:
    errors = []

    if not isinstance(user.get("id"), int):
        errors.append("id must be an integer")

    if not user.get("email"):
        errors.append("email is required")
    elif "@" not in user["email"]:
        errors.append("email must contain @")

    if not user.get("roles"):
        errors.append("at least one role is required")

    return errors


bad_users = [
    {"id": "1", "email": "no-at-symbol", "roles": []},
    {"id": 2, "email": "", "roles": ["reader"]},
]

all_errors = {
    index: errors
    for index, user in enumerate(bad_users)
    if (errors := validate_user(user))
}

print(all_errors)
print("valid:", not any(all_errors.values()))

{0: ['id must be an integer', 'email must contain @', 'at least one role is required'], 1: ['email is required']}
valid: False


## Problem 10 — Rolling aggregation with a fixed-size window

### Task

Write a generator `moving_average(values, window_size)`.

Requirements:

- Return a lazy iterator.
- Use a `deque`.
- Use `sum` to aggregate each current window.
- Only yield once the window is full.

In [25]:
# Solution

def moving_average(values: Iterable[float], window_size: int) -> Iterator[float]:
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    window: deque[float] = deque(maxlen=window_size)

    for value in values:
        window.append(value)
        if len(window) == window_size:
            yield sum(window) / window_size


print(list(moving_average([10, 20, 30, 40, 50], 3)))
print(list(moving_average(range(1, 11), 4)))

[20.0, 30.0, 40.0]
[2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5]


### Optimized version

The previous solution is simple. The next version avoids summing the whole window each time.

In [26]:
def moving_average_fast(values: Iterable[float], window_size: int) -> Iterator[float]:
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    window: deque[float] = deque()
    total = 0.0

    for value in values:
        window.append(value)
        total += value

        if len(window) > window_size:
            total -= window.popleft()

        if len(window) == window_size:
            yield total / window_size


print(list(moving_average_fast([10, 20, 30, 40, 50], 3)))

[20.0, 30.0, 40.0]


## Problem 11 — Infinite iterables and bounded aggregation

### Task

Use `itertools.count()` to make an infinite stream, but aggregate only the first `n` values.

Compute:

1. Sum of the first 10 integers starting at 1.
2. Whether any of the first 100 squares is greater than 5000.
3. Maximum of the first 20 multiples of 7.

In [27]:
# Solution

first_10_sum = sum(islice(count(1), 10))

any_square_gt_5000 = any(x * x > 5000 for x in islice(count(1), 100))

max_20_multiples_of_7 = max(islice((7 * x for x in count(1)), 20))

print("first_10_sum:", first_10_sum)
print("any_square_gt_5000:", any_square_gt_5000)
print("max_20_multiples_of_7:", max_20_multiples_of_7)

first_10_sum: 55
any_square_gt_5000: True
max_20_multiples_of_7: 140


### Warning

Never run `sum(count())`, `list(count())`, `all(count())`, or similar unbounded operations on infinite iterables.

## Problem 12 — Group-wise aggregation

### Task

Given products, compute:

1. Total inventory value per category.
2. Most expensive product per category.
3. Whether every category has at least one product with rating >= 4.5.

In [28]:
# Solution

inventory = [
    {"category": "books", "name": "Python Deep Dive", "price": Decimal("45.00"), "qty": 10, "rating": 4.8},
    {"category": "books", "name": "Algorithms", "price": Decimal("60.00"), "qty": 4, "rating": 4.6},
    {"category": "tools", "name": "Keyboard", "price": Decimal("90.00"), "qty": 7, "rating": 4.7},
    {"category": "tools", "name": "Mouse", "price": Decimal("35.00"), "qty": 12, "rating": 4.3},
    {"category": "games", "name": "Chess Set", "price": Decimal("30.00"), "qty": 5, "rating": 4.9},
]

by_category = defaultdict(list)

for item in inventory:
    by_category[item["category"]].append(item)

inventory_value = {
    category: sum(item["price"] * item["qty"] for item in items)
    for category, items in by_category.items()
}

most_expensive = {
    category: max(items, key=lambda item: item["price"])
    for category, items in by_category.items()
}

every_category_has_high_rating = all(
    any(item["rating"] >= 4.5 for item in items)
    for items in by_category.values()
)

print("inventory_value:", inventory_value)
print("most_expensive:", most_expensive)
print("every_category_has_high_rating:", every_category_has_high_rating)

inventory_value: {'books': Decimal('690.00'), 'tools': Decimal('1050.00'), 'games': Decimal('150.00')}
most_expensive: {'books': {'category': 'books', 'name': 'Algorithms', 'price': Decimal('60.00'), 'qty': 4, 'rating': 4.6}, 'tools': {'category': 'tools', 'name': 'Keyboard', 'price': Decimal('90.00'), 'qty': 7, 'rating': 4.7}, 'games': {'category': 'games', 'name': 'Chess Set', 'price': Decimal('30.00'), 'qty': 5, 'rating': 4.9}}
every_category_has_high_rating: True


## Challenge Problem 13 — Build your own `aggregate`

### Task

Write a function:

```python
aggregate(items, *, where=None, select=None, reducer=sum, default=None)
```

Behavior:

- `where` filters items.
- `select` transforms each item.
- `reducer` aggregates transformed items.
- `default` is returned when no items remain after filtering.

In [29]:
# Solution

def aggregate(
    items: Iterable[Any],
    *,
    where: Callable[[Any], bool] | None = None,
    select: Callable[[Any], Any] | None = None,
    reducer: Callable[[Iterable[Any]], Any] = sum,
    default: Any = None,
) -> Any:
    stream = items

    if where is not None:
        stream = (item for item in stream if where(item))

    if select is not None:
        stream = (select(item) for item in stream)

    stream = iter(stream)
    sentinel = object()
    first = next(stream, sentinel)

    if first is sentinel:
        return default

    def rebuilt() -> Iterator[Any]:
        yield first
        yield from stream

    return reducer(rebuilt())


numbers = [10, -5, 2, 0, 8]

print(aggregate(numbers, where=lambda x: x > 0))
print(aggregate(numbers, where=lambda x: x > 100, default="empty"))
print(aggregate(numbers, where=lambda x: x > 0, select=lambda x: x ** 2))
print(aggregate(numbers, where=lambda x: x > 0, reducer=max))

20
empty
168
10


### Extra: aggregate dictionaries

In [30]:
posted_total_usd = aggregate(
    parse_transactions("transactions.csv"),
    where=lambda row: row["status"] == "posted" and row["currency"] == "USD",
    select=lambda row: Decimal(row["amount"]),
    reducer=sum,
    default=Decimal("0"),
)

largest_posted = aggregate(
    parse_transactions("transactions.csv"),
    where=lambda row: row["status"] == "posted",
    reducer=lambda rows: max(rows, key=lambda row: abs(Decimal(row["amount"]))),
    default=None,
)

print("posted_total_usd:", posted_total_usd)
print("largest_posted:", largest_posted)

posted_total_usd: 315.25
largest_posted: {'id': '7', 'account': 'A400', 'amount': '5000.00', 'currency': 'GBP', 'status': 'posted'}


## Mini test suite

Run this cell after completing or modifying the solutions.

In [31]:
assert all_numeric([1, 2, Decimal("3")])
assert not all_numeric([1, True])
assert all_numeric([1, True], allow_bool=True)
assert smallest_positive([-3, 0, 8, 2]) == 2
assert smallest_positive([-3, 0]) is None
assert product_loop([2, 3, 4]) == 24
assert product_reduce([2, 3, 4]) == 24
assert first_match([1, 3, 4], lambda x: x % 2 == 0) == 4
assert list(moving_average([10, 20, 30], 2)) == [15, 25]
assert first_10_sum == 55
assert any_square_gt_5000 is True
assert max_20_multiples_of_7 == 140
assert posted_total_usd == Decimal("315.25")

print("All tests passed.")

All tests passed.


## Summary

You practiced advanced aggregator patterns:

- `min`, `max`, and `sum` for numeric and object aggregation
- `any` and `all` for expressive validation
- lazy generator expressions
- iterator exhaustion and ways to avoid bugs
- `key=` and `default=`
- streaming file processing
- custom reducers
- group-wise aggregation
- bounded aggregation over infinite iterables